In [83]:
import pandas as pd
import geopandas as gpd
collisions = pd.read_csv("data/raw/dft-road-casualty-statistics-collision-last-5-years.csv", low_memory=False)
collisions = collisions.rename(
    columns={
        "location_easting_osgr": "easting",
        "location_northing_osgr": "northing",
        "local_authority_ons_district": "borough_code"
    }
)
casualties = pd.read_csv("data/raw/dft-road-casualty-statistics-casualty-last-5-years.csv", low_memory=False)
casualties = casualties.rename(
    columns={
        "age_of_casualty": "age",
        "casualty_severity": "severity",
    }
)
collisions.head() # what is one row?
collisions.info() # column types and blanks
collisions.describe() # impossible numbers
collisions.isna().sum() # blanks per column
collisions.duplicated().sum() # repeated rows

<class 'pandas.DataFrame'>
RangeIndex: 513801 entries, 0 to 513800
Data columns (total 44 columns):
 #   Column                                            Non-Null Count   Dtype  
---  ------                                            --------------   -----  
 0   collision_index                                   513801 non-null  str    
 1   collision_year                                    513801 non-null  int64  
 2   collision_ref_no                                  513801 non-null  str    
 3   easting                                           513748 non-null  float64
 4   northing                                          513748 non-null  float64
 5   longitude                                         513748 non-null  float64
 6   latitude                                          513748 non-null  float64
 7   police_force                                      513801 non-null  int64  
 8   collision_severity                                513801 non-null  int64  
 9   number_of_vehic

np.int64(0)

In [84]:
casualties.info()

<class 'pandas.DataFrame'>
RangeIndex: 652821 entries, 0 to 652820
Data columns (total 24 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   collision_index                     652821 non-null  str    
 1   collision_year                      652821 non-null  int64  
 2   collision_ref_no                    652821 non-null  str    
 3   vehicle_reference                   652821 non-null  int64  
 4   casualty_reference                  652821 non-null  int64  
 5   casualty_class                      652821 non-null  int64  
 6   sex_of_casualty                     652821 non-null  int64  
 7   age                                 652821 non-null  int64  
 8   age_band_of_casualty                652821 non-null  int64  
 9   severity                            652821 non-null  int64  
 10  pedestrian_location                 652821 non-null  int64  
 11  pedestrian_movement                 6

In [85]:
"borough_code" in collisions.columns
collisions["borough_code"].dropna().sample(5, random_state=42)

87963     E09000006
483325    E06000054
333882    E09000003
146499    E08000032
197636    W06000019
Name: borough_code, dtype: str

In [86]:
import geopandas as gpd

# crs= tells GeoPandas what coordinate system the input numbers are already in. 
# .to_crs(27700) transforms those numbers into British National Grid eastings and northings.

# Create points from the collision coordinates.
# Pass easting first and northing second.
collision_points = gpd.GeoDataFrame(
    collisions,
    geometry=gpd.points_from_xy(
        collisions["easting"],
        collisions["northing"],
    ),
    crs="EPSG:27700",
)


# Use the same CRS for the boundary and the points before filtering. 
# This keeps collisions strictly inside the buffered boundary.
boundary = (
    gpd.read_file("data/clean/london_plus_400m.geojson")
    .to_crs(27700)
    .geometry.iloc[0]
)


# Filter the collisions to those strictly inside the buffered boundary.
collisions = collisions.loc[collision_points.within(boundary)].copy()


#### Filtering casualties to pedestrians aged 4 to 11 

In [87]:
is_pedestrian = casualties["casualty_class"].eq(3)
is_aged_4_to_11 = casualties["age"].between(4, 11)

child_pedestrians = casualties.loc[is_pedestrian & is_aged_4_to_11].copy()

print(f"Casualties kept: {len(child_pedestrians):,}")
print(child_pedestrians["age"].value_counts().sort_index())

london_child_pedestrians = child_pedestrians.merge(
    collisions,
    on="collision_index",
    how="inner",
    validate="many_to_one",
    suffixes=("", "_collision"),
)

print(f"Casualties within the boundary: {len(london_child_pedestrians):,}")

# Checking how many casualties were removed:
casualties_removed = len(casualties) - len(child_pedestrians)

print(f"Original casualty rows: {len(casualties):,}")
print(f"Casualties kept: {len(child_pedestrians):,}")
print(f"Casualties removed: {casualties_removed:,}")
print("\n")

# Checking how many casualties were kept and removed:
print(f"Before join: {len(child_pedestrians):,}")
print(f"After join: {len(london_child_pedestrians):,}")
print(f"Casualties dropped: {len(child_pedestrians) - len(london_child_pedestrians):,}")

Casualties kept: 11,167
age
4      974
5     1005
6     1047
7     1014
8     1052
9     1211
10    1669
11    3195
Name: count, dtype: int64
Casualties within the boundary: 1,890
Original casualty rows: 652,821
Casualties kept: 11,167
Casualties removed: 641,654


Before join: 11,167
After join: 1,890
Casualties dropped: 9,277


In [88]:
london_child_pedestrians["date"] = pd.to_datetime(
    london_child_pedestrians["date"],
    dayfirst=True,
    errors="coerce",
)

# Show dates in UK format while keeping the date column as a datetime.
preview = london_child_pedestrians[["date", "time"]].head().copy()
preview["date"] = preview["date"].dt.strftime("%d/%m/%Y")
print(preview)
print("Missing or unparseable dates:", london_child_pedestrians["date"].isna().sum())


         date   time
0  15/01/2021  14:05
1  21/01/2021  15:14
2  24/02/2022  15:38
3  28/03/2022  16:18
4  17/09/2022  11:00
Missing or unparseable dates: 0


#### Checking if the same collision_index + casualty_reference only appear once

In [89]:
key_columns = ["collision_index", "casualty_reference"]

duplicate_rows = london_child_pedestrians.loc[
    london_child_pedestrians.duplicated(subset=key_columns, keep=False)
]

print(f"Rows with a repeated key: {len(duplicate_rows):,}")
duplicate_rows.sort_values(key_columns)

Rows with a repeated key: 0


,collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age,age_band_of_casualty,severity,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight


In [90]:
"severity" in london_child_pedestrians.columns

True

In [91]:
columns_to_keep = [
    "collision_index",
    "date",
    "time",
    "easting",
    "northing",
    "borough_code",
    "age",
    "severity"
]

london_child_pedestrians_selected = london_child_pedestrians[columns_to_keep].copy()

london_child_pedestrians_selected.head()

,collision_index,date,time,easting,northing,borough_code,age,severity
0,2021010288881,2021-01-15,14:05,539362.0,186433.0,E09000031,7,2
1,2021010289554,2021-01-21,15:14,528309.0,184644.0,E09000007,11,3
2,2022010362058,2022-02-24,15:38,531264.0,167890.0,E09000008,4,2
3,2022010368028,2022-03-28,16:18,525301.0,162003.0,E09000029,11,3
4,2022010399851,2022-09-17,11:00,527835.0,178676.0,E09000020,7,3


#### Replace numeric codes in severity with words

In [98]:
severity_labels = {1: "fatal", 2: "serious", 3: "light"}

london_child_pedestrians_selected["severity"] = (
    london_child_pedestrians_selected["severity"].replace(severity_labels)
)

london_child_pedestrians_selected["severity"].value_counts(dropna=False)

london_child_pedestrians_selected.head()

,collision_index,date,time,easting,northing,borough_code,age,severity
0,2021010288881,2021-01-15,14:05,539362.0,186433.0,E09000031,7,serious
1,2021010289554,2021-01-21,15:14,528309.0,184644.0,E09000007,11,light
2,2022010362058,2022-02-24,15:38,531264.0,167890.0,E09000008,4,serious
3,2022010368028,2022-03-28,16:18,525301.0,162003.0,E09000029,11,light
4,2022010399851,2022-09-17,11:00,527835.0,178676.0,E09000020,7,light


#### Filtering out columns that contain the numeric value -1

In [92]:
counts = london_child_pedestrians_selected.eq(-1).sum()
counts[counts.gt(0)]

Series([], dtype: int64)

#### Filtering out columns that have missing values for northing and easting

In [93]:
coords = london_child_pedestrians_selected[["easting", "northing"]]
coords.isna().sum()

easting     0
northing    0
dtype: int64

In [100]:
london_child_pedestrians_selected.head(5)

,collision_index,date,time,easting,northing,borough_code,age,severity
0,2021010288881,2021-01-15,14:05,539362.0,186433.0,E09000031,7,serious
1,2021010289554,2021-01-21,15:14,528309.0,184644.0,E09000007,11,light
2,2022010362058,2022-02-24,15:38,531264.0,167890.0,E09000008,4,serious
3,2022010368028,2022-03-28,16:18,525301.0,162003.0,E09000029,11,light
4,2022010399851,2022-09-17,11:00,527835.0,178676.0,E09000020,7,light


In [99]:
london_child_pedestrians_selected["severity"].unique()
london_child_pedestrians_selected["severity"].value_counts(dropna=False)

severity
light      1515
serious     369
fatal         6
Name: count, dtype: int64

In [ ]:
output_path = "data/clean/child_ped_collisions_clean.csv"
london_child_pedestrians_selected.to_csv(output_path, index=False)
print(f"Exported {len(london_child_pedestrians_selected):,} rows to {output_path}")
